# Proyecto Final Integrador
## Precio esperado y deteccion de desvios en publicaciones de Airbnb de la Ciudad Autonoma de Buenos Aires

**Integrantes:** Ximena Romero y Carlos Siboni
**Diplomatura en Ciencia de Datos y Analisis Avanzado - Centro de e-Learning UTN BA**

Notebook ejecutado de punta a punta. Cada tabla y figura que cita el informe se
guarda en `results/`.

El flujo sigue CRISP-DM. Dos reglas se fijaron en la pre-entrega y no se tocan aca:

1. El **holdout del 20% agrupado por anfitrion** se aparta con semilla 42 y se evalua
   una sola vez, en la seccion 5. No participa del universo, ni de las grillas de
   hiperparametros, ni del umbral de desvio.
2. El **umbral de desvio** queda fijado en 0,4253 en escala logaritmica, equivalente a
   cobrar mas de 53,0% por encima o mas de 34,6% por debajo del precio esperado.

**Como ejecutar:** ver `README.md`. Los datos no se versionan; `data/README.md` indica
de donde bajarlos.


## 0. Configuracion, imports y funciones auxiliares

In [1]:
# =============================================================================
# Proyecto Final Integrador - Precio esperado y deteccion de desvios
# Airbnb CABA - Inside Airbnb, snapshot 2026-09-23
#
# Flujo CRISP-DM. Cada seccion guarda sus tablas en results/.
# El holdout del 20% agrupado por anfitrion se reconstruye con la misma semilla
# de la pre-entrega y se evalua UNA SOLA VEZ, en la seccion 5.
# =============================================================================
import json
import re
import time
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, KFold, cross_validate
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

# ------------------------------- configuracion -------------------------------
DATA_DIR = Path("data")
RES_DIR = Path("results")
SNAPSHOT = None   # se deriva de last_scraped al cargar los datos
SEED = 42
N_FOLDS, N_REP = 5, 4
ALFA = 0.05
UMBRAL_LOG = 0.4253          # fijado en la pre-entrega; NO se recalcula
ROJO, GRIS, AZUL = "#8B0000", "#555555", "#1f6feb"

RES_DIR.mkdir(exist_ok=True)
T0 = time.time()
BUF = []


def out(*a):
    s = " ".join(str(x) for x in a)
    print(s)
    BUF.append(s)


def titulo(s):
    out("\n" + "=" * 78)
    out(s)
    out("=" * 78)


def guardar(df, nombre, titulo_tabla=None, dec=3, index=True):
    """Imprime la tabla y la deja en results/ para que el informe la cite."""
    df.to_csv(RES_DIR / f"{nombre}.csv", index=index)
    if titulo_tabla:
        out(f"\n{titulo_tabla}")
    out(df.to_string(float_format=lambda v: f"{v:,.{dec}f}"))
    return df


def norm(s):
    return unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().upper().strip()


def parse_price(s):
    if not pd.api.types.is_numeric_dtype(s):
        s = s.astype(str).str.replace(r"[$,\s]", "", regex=True).replace({"nan": np.nan, "": np.nan})
    return pd.to_numeric(s, errors="coerce")


def parse_banos(s):
    if pd.isna(s):
        return np.nan
    s = str(s).lower()
    m = re.search(r"\d+(\.\d+)?", s)
    if m:
        return float(m.group())
    return 0.5 if "half" in s else np.nan


def contar_amenities(s):
    try:
        return len(json.loads(s))
    except Exception:
        return np.nan


def buscar_csv(*patrones):
    for p in patrones:
        for d in (DATA_DIR, Path(".")):
            hits = sorted(d.glob(p))
            if hits:
                return hits[0]
    return None

## 1. Comprension del negocio

En CABA conviven casi 30.000 publicaciones de Airbnb y cada anfitrion fija su precio
mirando lo que cobran sus vecinos, sin controlar por capacidad, dormitorios, banos ni
equipamiento. El proyecto estima el **precio de consenso del mercado** para un conjunto
de atributos y mide la brecha entre ese precio y el que cada anfitrion publica.

**Audiencia:** duenos particulares e inversores de alquiler temporario de CABA.

**KPI de negocio:** la proporcion de publicaciones con un desvio superior al umbral
fijado de antemano, y la diferencia de actividad reciente entre los extremos de la brecha.

**Metrica tecnica:** MdAPE, el error porcentual mediano, que no se deja arrastrar por
las publicaciones de precio bajo. Se reportan tambien MAPE y R2 sobre log(precio).

**Benchmark de negocio:** el precio mediano por barrio y tipo de habitacion, que es la
heuristica que usa hoy cualquier anfitrion.

In [2]:
titulo("1. Comprension del negocio")
out("Problema: los anfitriones fijan precio comparando informalmente, sin controlar")
out("por los atributos de la unidad. El modelo estima el precio de consenso del mercado")
out("para un conjunto de atributos y la brecha senala publicaciones candidatas a revision.")
out("")
out("Metrica principal: MdAPE (error porcentual mediano), robusto a precios bajos.")
out("Benchmark de negocio: precio mediano por barrio y tipo de habitacion.")
out("Criterio de exito (fijado en la pre-entrega): el modelo final debe superar al")
out("benchmark y a la regresion lineal, ambos con diferencia significativa en test t")
out("pareado unilateral con correccion de Nadeau y Bengio (2003).")
out(f"Umbral de desvio fijado ex ante: {UMBRAL_LOG} en escala logaritmica "
    f"(+{np.expm1(UMBRAL_LOG)*100:.1f}% / {np.expm1(-UMBRAL_LOG)*100:.1f}%).")


1. Comprension del negocio
Problema: los anfitriones fijan precio comparando informalmente, sin controlar
por los atributos de la unidad. El modelo estima el precio de consenso del mercado
para un conjunto de atributos y la brecha senala publicaciones candidatas a revision.

Metrica principal: MdAPE (error porcentual mediano), robusto a precios bajos.
Benchmark de negocio: precio mediano por barrio y tipo de habitacion.
Criterio de exito (fijado en la pre-entrega): el modelo final debe superar al
benchmark y a la regresion lineal, ambos con diferencia significativa en test t
pareado unilateral con correccion de Nadeau y Bengio (2003).
Umbral de desvio fijado ex ante: 0.4253 en escala logaritmica (+53.0% / -34.6%).


## 2. Comprension de los datos

Fuente: Inside Airbnb, snapshot de CABA del 23 de Septiembre de 2026, bajo licencia
Creative Commons Attribution 4.0. Se usa el archivo detallado `listings.csv.gz`.

Esta seccion documenta el diccionario de las diez variables principales, los nulos y
la concentracion de la oferta por anfitrion. Ese ultimo dato es el que obliga a validar
agrupando por anfitrion en la seccion 4.

In [3]:
titulo("2. Comprension de los datos")
path_listings = buscar_csv("listings.csv.gz", "listings*.csv.gz", "listings*detallad*.csv", "listings.csv")
if path_listings is None:
    raise FileNotFoundError(f"No encontre listings.csv.gz en {DATA_DIR.resolve()} ni en el directorio actual.")
det = pd.read_csv(path_listings, low_memory=False)
det["price"] = parse_price(det["price"])
SNAPSHOT = pd.to_datetime(det["last_scraped"]).max()
out(f"archivo: {path_listings} | filas: {len(det):,} | columnas: {det.shape[1]}")
out(f"anfitriones unicos: {det['host_id'].nunique():,}")
out(f"snapshot (last_scraped maximo): {SNAPSHOT.date()}")

# --- 2.1 calidad ---
VARS10 = ["price", "neighbourhood_cleansed", "room_type", "accommodates", "bedrooms",
          "bathrooms_text", "amenities", "minimum_nights", "calculated_host_listings_count",
          "number_of_reviews_ltm"]
filas = []
for c in VARS10:
    s = det[c]
    num = pd.api.types.is_numeric_dtype(s)
    filas.append({"variable": c, "tipo": "numerica" if num else "categorica/texto",
                  "nulos_pct": s.isna().mean() * 100, "unicos": s.nunique(),
                  "mediana": s.median() if num else np.nan,
                  "minimo": s.min() if num else np.nan,
                  "maximo": s.max() if num else np.nan})
guardar(pd.DataFrame(filas).set_index("variable"), "eda_diccionario",
        "2.1 Diccionario de las 10 variables principales", dec=2)

otras = ["license", "neighbourhood_group_cleansed", "review_scores_rating", "first_review",
         "last_review", "reviews_per_month", "beds", "availability_365"]
otras = [c for c in otras if c in det.columns]
guardar(pd.DataFrame({"nulos_pct": det[otras].isna().mean() * 100}).round(2),
        "eda_calidad", "Nulos en otras variables relevantes", dec=2)

# bedrooms: el nulo contiene informacion
nb = det["bedrooms"].isna()
patron = r"monoambiente|mono ambiente|monoamb|1 ambiente|un ambiente|studio|estudio|loft"
nombre = det["name"].fillna("").str.lower() if "name" in det.columns else pd.Series("", index=det.index)
bed = pd.DataFrame({
    "bedrooms_nulo": [det.loc[nb, "accommodates"].median(), (det.loc[nb, "accommodates"] <= 2).mean() * 100,
                      nombre[nb].str.contains(patron).mean() * 100],
    "bedrooms_informado": [det.loc[~nb, "accommodates"].median(), (det.loc[~nb, "accommodates"] <= 2).mean() * 100,
                           nombre[~nb].str.contains(patron).mean() * 100]},
    index=["accommodates_mediana", "pct_capacidad_hasta_2", "pct_monoambiente_en_titulo"])
guardar(bed, "eda_bedrooms", f"bedrooms nulos: {nb.sum():,} ({nb.mean()*100:.2f}%)", dec=1)
guardar(det.groupby("room_type")["bedrooms"].apply(lambda s: s.isna().mean() * 100).to_frame("nulos_pct"),
        "eda_bedrooms_por_tipo", "bedrooms nulos por tipo de habitacion", dec=2)

# concentracion por anfitrion
ph = det.groupby("host_id").size()
guardar(pd.DataFrame({
    "pct_anfitriones": [(ph > 1).mean() * 100, (ph > 2).mean() * 100],
    "pct_publicaciones": [(det["calculated_host_listings_count"] > 1).mean() * 100,
                          (det["calculated_host_listings_count"] > 2).mean() * 100]},
    index=["mas_de_1_publicacion", "mas_de_2_publicaciones"]),
    "eda_concentracion", "2.2 Concentracion por anfitrion", dec=1)

# barrios
bar_n = det["neighbourhood_cleansed"].map(norm)
out(f"\nbarrios con oferta: {det['neighbourhood_cleansed'].nunique()}")
guardar((det["neighbourhood_cleansed"].value_counts().head(10) / len(det) * 100).to_frame("pct_publicaciones"),
        "eda_barrios_top10", "Diez barrios con mas oferta (% del total)", dec=1)


2. Comprension de los datos
archivo: data\listings.csv | filas: 29,753 | columnas: 90
anfitriones unicos: 14,623
snapshot (last_scraped maximo): 2026-09-27

2.1 Diccionario de las 10 variables principales
                                            tipo  nulos_pct  unicos    mediana  minimo         maximo
variable                                                                                             
price                                   numerica       7.12   11775 106,393.00  246.59 160,563,760.00
neighbourhood_cleansed          categorica/texto       0.00      47        NaN     NaN            NaN
room_type                       categorica/texto       0.00       4        NaN     NaN            NaN
accommodates                            numerica       0.00      16       2.00    1.00          16.00
bedrooms                                numerica       0.04      20       1.00    0.00          31.00
bathrooms_text                  categorica/texto       0.07      48        NaN  

,pct_publicaciones
neighbourhood_cleansed,
Palermo,32.628642
Recoleta,13.722986
San Nicolas,8.130273
Retiro,4.937317
Monserrat,4.880180
Belgrano,4.826404
Balvanera,4.023124
Almagro,3.646691
Villa Crespo,3.434948


## 3. Preparacion de los datos

Tres decisiones, todas registradas en `data_log.md`:

- **Universo:** publicaciones con precio valido, estadia minima de hasta 90 noches y
  precio dentro de los percentiles 0,5 y 99,5. El recorte define la poblacion de estudio,
  no es una transformacion que el modelo aprenda, por eso se calcula sobre el total.
- **Variables derivadas:** cantidad de banos y si son compartidos a partir del texto de
  `bathrooms_text`, y cantidad de amenities a partir de la lista JSON.
- **Particion:** holdout del 20% **agrupado por anfitrion**. Un anfitrion con veinte
  publicaciones casi identicas no puede quedar a ambos lados de la particion.

In [4]:
titulo("3. Preparacion de los datos")
d = det.copy()
d["banos"] = d["bathrooms_text"].map(parse_banos)
d["bano_compartido"] = d["bathrooms_text"].str.contains("shared", case=False, na=False).astype(int)
d["n_amenities"] = d["amenities"].map(contar_amenities)

embudo = [("detallado completo", len(d))]
d = d[d["price"].notna()];        embudo.append(("con precio", len(d)))
d = d[d["price"] > 0];            embudo.append(("precio mayor a cero", len(d)))
d = d[d["minimum_nights"] <= 90]; embudo.append(("minimum_nights hasta 90", len(d)))
sin_recorte = d.reset_index(drop=True)
lo, hi = d["price"].quantile([0.005, 0.995])
u = d[d["price"].between(lo, hi)].reset_index(drop=True)
embudo.append((f"precio entre p0,5 y p99,5 ({lo:,.0f} a {hi:,.0f} ARS)", len(u)))
guardar(pd.DataFrame(embudo, columns=["paso", "publicaciones"]).set_index("paso"),
        "prep_embudo", "3.1 Universo de analisis", dec=0)

NUM = ["latitude", "longitude", "accommodates", "bedrooms", "beds", "banos", "bano_compartido",
       "minimum_nights", "n_amenities", "calculated_host_listings_count"]
CAT = ["neighbourhood_cleansed", "room_type"]

# --- particion holdout: misma semilla que la pre-entrega ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
i_dev, i_hold = next(gss.split(u, groups=u["host_id"]))
dev = u.iloc[i_dev].reset_index(drop=True)
hold = u.iloc[i_hold].reset_index(drop=True)
assert not (set(dev["host_id"]) & set(hold["host_id"])), "hay anfitriones en ambos conjuntos"
pd.Series(sorted(set(hold["host_id"])), name="host_id").to_csv(RES_DIR / "holdout_host_ids.csv", index=False)

guardar(pd.DataFrame({
    "publicaciones": [len(dev), len(hold)], "anfitriones": [dev["host_id"].nunique(), hold["host_id"].nunique()],
    "pct_Palermo": [(dev["neighbourhood_cleansed"].map(norm) == "PALERMO").mean() * 100,
                    (hold["neighbourhood_cleansed"].map(norm) == "PALERMO").mean() * 100],
    "precio_mediano": [dev["price"].median(), hold["price"].median()]},
    index=["desarrollo_80", "holdout_20"]), "prep_particion",
    "3.2 Particion agrupada por anfitrion (semilla 42)", dec=1)

# verificacion contra la pre-entrega
ref = Path("holdout_host_ids_preentrega.csv")
if ref.exists():
    previos = set(pd.read_csv(ref)["host_id"])
    comunes = previos & set(hold["host_id"])
    out(f"\nComparacion con el holdout de la pre-entrega: {len(previos):,} anfitriones registrados "
        f"entonces, {len(comunes):,} de ellos vuelven a caer en el holdout ahora.")
    out("La regla de particion y la semilla son las mismas; el universo cambio porque el")
    out("snapshot es posterior, asi que el conjunto no es identico.")

# gemelos
clave = ["host_id", "neighbourhood_cleansed", "room_type", "accommodates", "bedrooms", "banos"]
en_grupo = dev.duplicated(clave, keep=False)
cv_gem = (dev[en_grupo].groupby(clave, dropna=False)["price"]
          .agg(lambda s: s.std() / s.mean()).dropna())
out(f"\npublicaciones de desarrollo con al menos una gemela del mismo anfitrion: "
    f"{en_grupo.mean()*100:.1f}% | variacion mediana del precio dentro del grupo: {cv_gem.median()*100:.1f}%")

Xd, yd, pd_dev = dev[NUM + CAT], np.log(dev["price"].to_numpy()), dev["price"].to_numpy()
hosts_dev = dev["host_id"].to_numpy()


def prepro(escalar, num=NUM):
    pnum = (make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler())
            if escalar else SimpleImputer(strategy="median", add_indicator=True))
    return ColumnTransformer([("num", pnum, num),
                              ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT)])


class BenchmarkGrupo(BaseEstimator, RegressorMixin):
    """Precio de referencia por barrio y tipo, con respaldo por tipo y global."""
    def __init__(self, stat="median"):
        self.stat = stat

    def fit(self, X, y):
        p = np.exp(np.asarray(y))
        df = pd.DataFrame({"b": X["neighbourhood_cleansed"].to_numpy(),
                           "t": X["room_type"].to_numpy(), "p": p})
        self.tabla_ = df.groupby(["b", "t"])["p"].agg(self.stat).to_dict()
        self.tipo_ = df.groupby("t")["p"].agg(self.stat).to_dict()
        self.global_ = float(df["p"].agg(self.stat))
        return self

    def predict(self, X):
        v = [self.tabla_.get((b, t), self.tipo_.get(t, self.global_))
             for b, t in zip(X["neighbourhood_cleansed"], X["room_type"])]
        return np.log(np.asarray(v, dtype=float))


def metricas(precio, pred_log, y_log):
    p = np.exp(pred_log)
    ape = np.abs(precio - p) / precio
    return {"MdAPE": np.median(ape) * 100, "MAPE": ape.mean() * 100,
            "MAE_ARS": np.abs(precio - p).mean(), "R2_log": r2_score(y_log, pred_log)}


def scorer(est, X, y):
    pl = est.predict(X)
    real = np.exp(y)
    ape = np.abs(real - np.exp(pl)) / real
    return {"MdAPE": np.median(ape) * 100, "MAPE": ape.mean() * 100, "R2log": r2_score(y, pl)}


def folds_por_host(hosts, seed, n_folds):
    unicos = np.unique(hosts)
    perm = np.random.default_rng(seed).permutation(unicos)
    fold_de = pd.Series(np.arange(len(perm)) % n_folds, index=perm)
    f = fold_de.reindex(hosts).to_numpy()
    return [(np.where(f != k)[0], np.where(f == k)[0]) for k in range(n_folds)]


3. Preparacion de los datos

3.1 Universo de analisis
                                                    publicaciones
paso                                                             
detallado completo                                          29753
con precio                                                  27634
precio mayor a cero                                         27634
minimum_nights hasta 90                                     27563
precio entre p0,5 y p99,5 (22,123 a 1,596,087 ARS)          27287

3.2 Particion agrupada por anfitrion (semilla 42)
               publicaciones  anfitriones  pct_Palermo  precio_mediano
desarrollo_80          21566        10745         32.2       106,734.0
holdout_20              5721         2687         35.3       105,488.5

publicaciones de desarrollo con al menos una gemela del mismo anfitrion: 31.3% | variacion mediana del precio dentro del grupo: 9.4%


## 4. Modelado

Primero una busqueda de hiperparametros con GroupKFold de 3 folds sobre desarrollo, y
despues la comparacion con validacion cruzada agrupada de 5 folds por 4 repeticiones.

Se evaluan siete alternativas: los dos benchmarks de grupo, una regresion lineal, KNN,
Random Forest, Gradient Boosting por histogramas y XGBoost.

El **criterio de exito** se fijo en la pre-entrega: el modelo final debe superar al
benchmark de mediana **y** a la regresion lineal, ambos con diferencia significativa en
un test t pareado unilateral con la correccion de Nadeau y Bengio (2003), que compensa
el solapamiento entre los conjuntos de entrenamiento de los folds. Es un criterio
comparativo, asi que no puede calibrarse mirando los resultados.

Se reporta ademas el mismo error con particion aleatoria. La diferencia entre ambos
esquemas mide cuanto infla las metricas la fuga entre publicaciones del mismo anfitrion.

In [5]:
titulo("4. Modelado: busqueda de hiperparametros y validacion cruzada agrupada")
cv_busqueda = list(GroupKFold(n_splits=3).split(Xd, yd, groups=hosts_dev))


def buscar(nombre, hacer, grilla):
    """Grilla manual sobre GroupKFold de 3 folds; devuelve la mejor configuracion."""
    filas = []
    for params in grilla:
        sc = cross_validate(hacer(params), Xd, yd, cv=cv_busqueda, scoring=scorer, n_jobs=-1)
        filas.append({**params, "MdAPE": sc["test_MdAPE"].mean()})
    tab = pd.DataFrame(filas).sort_values("MdAPE").reset_index(drop=True)
    mejor = {k: v for k, v in tab.iloc[0].items() if k != "MdAPE"}
    for k, v in list(mejor.items()):
        if isinstance(v, (int, float, np.integer, np.floating)) and float(v).is_integer():
            mejor[k] = int(v)
        elif isinstance(v, np.floating):
            mejor[k] = float(v)
    out(f"  {nombre}: mejor {mejor} -> MdAPE {tab.iloc[0]['MdAPE']:.2f}  "
        f"(margen con el segundo: {tab.iloc[1]['MdAPE'] - tab.iloc[0]['MdAPE']:.3f} p.p.)")
    tab.insert(0, "modelo", nombre)
    return mejor, tab


tablas_grilla, mejores = [], {}
mejores["KNN"], t = buscar("KNN", lambda p: make_pipeline(prepro(True), KNeighborsRegressor(**p)),
                           [{"n_neighbors": k, "weights": w} for k in (5, 10, 20) for w in ("uniform", "distance")])
tablas_grilla.append(t)
mejores["Random Forest"], t = buscar(
    "Random Forest", lambda p: make_pipeline(prepro(False), RandomForestRegressor(
        n_estimators=200, random_state=SEED, n_jobs=1, **p)),
    [{"max_features": mf, "min_samples_leaf": ml} for mf in ("sqrt", 0.3) for ml in (1, 5)])
tablas_grilla.append(t)
mejores["Gradient Boosting"], t = buscar(
    "Gradient Boosting", lambda p: make_pipeline(prepro(False), HistGradientBoostingRegressor(
        random_state=SEED, **p)),
    [{"learning_rate": lr, "max_leaf_nodes": ml} for lr in (0.05, 0.1) for ml in (31, 63)])
tablas_grilla.append(t)
mejores["XGBoost"], t = buscar(
    "XGBoost", lambda p: make_pipeline(prepro(False), XGBRegressor(
        random_state=SEED, n_jobs=1, tree_method="hist", **p)),
    [{"max_depth": md, "learning_rate": lr, "n_estimators": 400}
     for md in (6, 8) for lr in (0.05, 0.1)])
tablas_grilla.append(t)
guardar(pd.concat(tablas_grilla, ignore_index=True), "modelado_grillas",
        "4.1 Busqueda de hiperparametros (GroupKFold de 3 folds sobre desarrollo)", dec=3, index=False)

MODELOS = {
    "Benchmark mediana": BenchmarkGrupo("median"),
    "Benchmark promedio": BenchmarkGrupo("mean"),
    "Regresion lineal": make_pipeline(prepro(True), LinearRegression()),
    "KNN": make_pipeline(prepro(True), KNeighborsRegressor(**mejores["KNN"])),
    "Random Forest": make_pipeline(prepro(False), RandomForestRegressor(
        n_estimators=200, random_state=SEED, n_jobs=1, **mejores["Random Forest"])),
    "Gradient Boosting": make_pipeline(prepro(False), HistGradientBoostingRegressor(
        random_state=SEED, **mejores["Gradient Boosting"])),
    "XGBoost": make_pipeline(prepro(False), XGBRegressor(
        random_state=SEED, n_jobs=1, tree_method="hist", **mejores["XGBoost"])),
}

splits_rep = []
for r in range(N_REP):
    splits_rep += folds_por_host(hosts_dev, SEED + r, N_FOLDS)
splits_alea = list(KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(Xd))
n_tr = np.mean([len(a) for a, _ in splits_rep])
n_te = np.mean([len(b) for _, b in splits_rep])

por_fold, filas = {}, []
for nm, mod in MODELOS.items():
    t = time.time()
    rep = cross_validate(mod, Xd, yd, cv=splits_rep, scoring=scorer, n_jobs=-1, error_score="raise")
    ale = cross_validate(mod, Xd, yd, cv=splits_alea, scoring=scorer, n_jobs=-1, error_score="raise")
    por_fold[nm] = rep["test_MdAPE"]
    filas.append({"modelo": nm, "MdAPE_agrupado": rep["test_MdAPE"].mean(),
                  "desvio": rep["test_MdAPE"].std(ddof=1), "MAPE_agrupado": rep["test_MAPE"].mean(),
                  "R2log_agrupado": rep["test_R2log"].mean(),
                  "MdAPE_aleatorio": ale["test_MdAPE"].mean(),
                  "brecha_pp": rep["test_MdAPE"].mean() - ale["test_MdAPE"].mean(),
                  "segundos": time.time() - t})
    out(f"  {nm:20s} {time.time()-t:6.1f} s")
cv = pd.DataFrame(filas).set_index("modelo")
bench_md = cv.loc["Benchmark mediana", "MdAPE_agrupado"]
cv["reduccion_vs_benchmark_pct"] = (bench_md - cv["MdAPE_agrupado"]) / bench_md * 100
guardar(cv, "modelado_cv_agrupada",
        f"4.2 Validacion cruzada agrupada por anfitrion ({N_FOLDS} folds x {N_REP} repeticiones)", dec=2)

# --- criterio de exito: test t pareado con correccion de Nadeau y Bengio ---
def test_pareado(err_ref, err_mod, alternativa="greater"):
    dif = np.asarray(err_ref) - np.asarray(err_mod)
    J = len(dif)
    var = dif.var(ddof=1)
    if var == 0:
        return dif.mean(), np.nan, np.nan
    t_c = dif.mean() / np.sqrt((1 / J + n_te / n_tr) * var)
    p_c = stats.t.sf(t_c, J - 1) if alternativa == "greater" else 2 * stats.t.sf(abs(t_c), J - 1)
    p_s = stats.ttest_rel(err_ref, err_mod, alternative=alternativa).pvalue
    return dif.mean(), p_c, p_s


filas = []
for cand in ["XGBoost", "Gradient Boosting", "Random Forest", "KNN"]:
    cumple = []
    for ref in ["Benchmark mediana", "Regresion lineal"]:
        mej, p_c, p_s = test_pareado(por_fold[ref], por_fold[cand])
        ok = bool(p_c < ALFA and mej > 0)
        cumple.append(ok)
        filas.append({"candidato": cand, "referencia": ref, "mejora_MdAPE_pp": mej,
                      "p_corregido": p_c, "p_sin_corregir": p_s, "supera": "si" if ok else "no"})
    filas.append({"candidato": cand, "referencia": "CRITERIO DE EXITO", "mejora_MdAPE_pp": np.nan,
                  "p_corregido": np.nan, "p_sin_corregir": np.nan,
                  "supera": "CUMPLE" if all(cumple) else "NO CUMPLE"})
guardar(pd.DataFrame(filas).set_index(["candidato", "referencia"]), "modelado_criterio_exito",
        "4.3 Criterio de exito: superar al benchmark y a la regresion lineal", dec=4)

candidatos = [c for c in ["XGBoost", "Gradient Boosting", "Random Forest", "KNN"]
              if cv.loc[c, "MdAPE_agrupado"] == cv.loc[["XGBoost", "Gradient Boosting", "Random Forest", "KNN"],
                                                       "MdAPE_agrupado"].min()]
GANADOR = candidatos[0]
out(f"\nModelo con menor MdAPE agrupado: {GANADOR} ({cv.loc[GANADOR,'MdAPE_agrupado']:.2f}%)")
rivales = [c for c in ["XGBoost", "Gradient Boosting", "Random Forest"] if c != GANADOR]
filas = []
for c in rivales:
    mej, p_c, p_s = test_pareado(por_fold[c], por_fold[GANADOR], "two-sided")
    filas.append({"comparacion": f"{GANADOR} vs {c}", "diferencia_MdAPE_pp": mej,
                  "p_corregido": p_c, "p_sin_corregir": p_s,
                  "veredicto": "diferencia significativa" if p_c < ALFA else "empate estadistico"})
guardar(pd.DataFrame(filas).set_index("comparacion"), "modelado_ganador_vs_rivales",
        "4.4 El ganador frente a los demas modelos de arboles (bilateral)", dec=4)

fig, ax = plt.subplots(figsize=(9, 4))
orden = [m for m in MODELOS if m != "Benchmark promedio"]
x, w = np.arange(len(orden)), 0.38
ax.bar(x - w / 2, cv.loc[orden, "MdAPE_aleatorio"], w, color=GRIS, label="Partición aleatoria")
ax.bar(x + w / 2, cv.loc[orden, "MdAPE_agrupado"], w, color=ROJO, label="Partición agrupada por anfitrión")
ax.set_xticks(x)
ax.set_xticklabels([m.replace("Regresion", "Regresión").replace("mediana", "mediana\n(barrio y tipo)")
                    for m in orden], rotation=18, ha="right", fontsize=9)
ax.set_ylabel("Error porcentual mediano (%)")
ax.set_title("Error de validación según el esquema de partición")
ax.legend()
plt.tight_layout(); plt.savefig(RES_DIR / "fig_validacion.png", dpi=150); plt.close()


4. Modelado: busqueda de hiperparametros y validacion cruzada agrupada
  KNN: mejor {'n_neighbors': 20, 'weights': 'uniform'} -> MdAPE 22.36  (margen con el segundo: 0.006 p.p.)
  Random Forest: mejor {'max_features': 0.3, 'min_samples_leaf': 5} -> MdAPE 21.20  (margen con el segundo: 0.214 p.p.)
  Gradient Boosting: mejor {'learning_rate': 0.05, 'max_leaf_nodes': 31} -> MdAPE 21.05  (margen con el segundo: 0.008 p.p.)
  XGBoost: mejor {'max_depth': 6, 'learning_rate': 0.05, 'n_estimators': 400} -> MdAPE 20.99  (margen con el segundo: 0.154 p.p.)

4.1 Busqueda de hiperparametros (GroupKFold de 3 folds sobre desarrollo)
               modelo  n_neighbors   weights  MdAPE max_features  min_samples_leaf  learning_rate  max_leaf_nodes  max_depth  n_estimators
0                 KNN       20.000   uniform 22.364          NaN               NaN            NaN             NaN        NaN           NaN
1                 KNN       20.000  distance 22.370          NaN               NaN            

## 5. Evaluacion sobre el holdout

Unica evaluacion sobre el 20% reservado. Se comparan el benchmark de negocio, la
regresion lineal y el modelo ganador de la seccion anterior.

In [6]:
titulo("5. Evaluacion sobre el holdout (una sola vez)")
out("El holdout no participo de ninguna decision: ni del universo, ni de las grillas,")
out("ni del umbral de desvio. Se evalua aqui por unica vez.")
Xh, ph_real = hold[NUM + CAT], hold["price"].to_numpy()
yh = np.log(ph_real)
filas, pred_hold = [], {}
for nm in ["Benchmark mediana", "Regresion lineal", GANADOR]:
    mod = clone(MODELOS[nm]).fit(Xd, yd)
    pl = mod.predict(Xh)
    pred_hold[nm] = pl
    filas.append({"modelo": nm, **metricas(ph_real, pl, yh)})
hm = pd.DataFrame(filas).set_index("modelo")
hm["reduccion_vs_benchmark_pct"] = ((hm.loc["Benchmark mediana", "MdAPE"] - hm["MdAPE"])
                                    / hm.loc["Benchmark mediana", "MdAPE"] * 100)
guardar(hm, "holdout_metricas", f"5.1 Resultados sobre el holdout ({len(hold):,} publicaciones)", dec=2)
out(f"\nDesarrollo vs holdout para {GANADOR}: MdAPE {cv.loc[GANADOR,'MdAPE_agrupado']:.2f}% en validacion "
    f"cruzada agrupada contra {hm.loc[GANADOR,'MdAPE']:.2f}% en el holdout.")


5. Evaluacion sobre el holdout (una sola vez)
El holdout no participo de ninguna decision: ni del universo, ni de las grillas,
ni del umbral de desvio. Se evalua aqui por unica vez.

5.1 Resultados sobre el holdout (5,721 publicaciones)
                   MdAPE  MAPE   MAE_ARS  R2_log  reduccion_vs_benchmark_pct
modelo                                                                      
Benchmark mediana  28.34 36.31 55,000.73    0.20                        0.00
Regresion lineal   24.45 34.51 46,104.10    0.43                       13.72
XGBoost            21.51 30.41 42,123.42    0.51                       24.12

Desarrollo vs holdout para XGBoost: MdAPE 20.98% en validacion cruzada agrupada contra 21.51% en el holdout.


## 6. Robustez

Dos chequeos sobre las decisiones que podrian haber inclinado el resultado: la eleccion
de la mediana en lugar del promedio como benchmark, y la forma de imputar `bedrooms`,
cuyos nulos no son aleatorios.

In [7]:
titulo("6. Robustez")
# 6.1 benchmark promedio vs mediana, con y sin recorte
filas = []
for nombre_u, uu in [("con recorte", dev),
                     ("sin recorte", sin_recorte[sin_recorte["host_id"].isin(set(dev["host_id"]))].reset_index(drop=True))]:
    sp = folds_por_host(uu["host_id"].to_numpy(), SEED, N_FOLDS)
    for stat in ["median", "mean"]:
        sc = cross_validate(BenchmarkGrupo(stat), uu[CAT], np.log(uu["price"].to_numpy()),
                            cv=sp, scoring=scorer, error_score="raise")
        filas.append({"universo": nombre_u, "benchmark": "mediana" if stat == "median" else "promedio",
                      "MdAPE": sc["test_MdAPE"].mean(), "MAPE": sc["test_MAPE"].mean()})
guardar(pd.DataFrame(filas).set_index(["universo", "benchmark"]), "robustez_benchmark",
        "6.1 Benchmark por promedio y por mediana, con y sin recorte de extremos", dec=2)

# 6.2 sensibilidad de la imputacion de bedrooms
dev2 = dev.copy()
es_entera = dev2["room_type"].eq("Entire home/apt")
dev2["bedrooms"] = dev2["bedrooms"].fillna(pd.Series(np.where(es_entera, 0.0, 1.0), index=dev2.index))
sp5 = folds_por_host(hosts_dev, SEED, N_FOLDS)
filas = []
for etiqueta, Xv in [("mediana + indicador (decision tomada)", Xd),
                     ("0 en unidades completas, 1 en habitaciones", dev2[NUM + CAT])]:
    sc = cross_validate(clone(MODELOS[GANADOR]), Xv, yd, cv=sp5, scoring=scorer, n_jobs=-1)
    filas.append({"imputacion_bedrooms": etiqueta, "MdAPE": sc["test_MdAPE"].mean(),
                  "MAPE": sc["test_MAPE"].mean(), "R2log": sc["test_R2log"].mean()})
guardar(pd.DataFrame(filas).set_index("imputacion_bedrooms"), "robustez_bedrooms",
        f"6.2 Sensibilidad de la imputacion de bedrooms ({GANADOR}, CV agrupada de {N_FOLDS} folds)", dec=2)


6. Robustez

6.1 Benchmark por promedio y por mediana, con y sin recorte de extremos
                       MdAPE   MAPE
universo    benchmark              
con recorte mediana    28.11  35.76
            promedio   39.19  51.61
sin recorte mediana    28.38  36.43
            promedio   83.80 200.03

6.2 Sensibilidad de la imputacion de bedrooms (XGBoost, CV agrupada de 5 folds)
                                            MdAPE  MAPE  R2log
imputacion_bedrooms                                           
mediana + indicador (decision tomada)       21.03 28.87   0.51
0 en unidades completas, 1 en habitaciones  21.14 28.79   0.51


,MdAPE,MAPE,R2log
imputacion_bedrooms,,,
mediana + indicador (decision tomada),21.025058,28.872094,0.511943
"0 en unidades completas, 1 en habitaciones",21.141837,28.793109,0.513928


## 7. Interpretabilidad

Que atributos explican el precio esperado, medido con SHAP sobre el holdout y agrupando
las columnas one-hot por variable original.

**SHAP explica el comportamiento del modelo, no relaciones causales.**

In [8]:
titulo("7. Interpretabilidad: que explica el precio esperado")
import shap  # noqa: E402

mod_final = clone(MODELOS[GANADOR]).fit(Xd, yd)
pre = mod_final.named_steps[[k for k in mod_final.named_steps if "columntransformer" in k][0]]
est = mod_final.named_steps[list(mod_final.named_steps)[-1]]
nombres = list(pre.get_feature_names_out())
Xh_t = pre.transform(Xh)
muestra = min(2000, len(Xh_t))
rng = np.random.default_rng(SEED)
idx = rng.choice(len(Xh_t), muestra, replace=False)
expl = shap.TreeExplainer(est)
sv = expl.shap_values(Xh_t[idx])

imp = pd.Series(np.abs(sv).mean(0), index=nombres)
agrupado = {}
for n, v in imp.items():
    base = n.split("__", 1)[-1]
    base = base.split("_")[0] if base.startswith(("neighbourhood_cleansed", "room_type")) else base
    base = "neighbourhood_cleansed" if base.startswith("neighbourhood") else ("room_type" if base.startswith("room") else base)
    agrupado[base] = agrupado.get(base, 0) + v
sg = pd.Series(agrupado).sort_values(ascending=False).to_frame("shap_medio_abs")
sg["pct"] = sg["shap_medio_abs"] / sg["shap_medio_abs"].sum() * 100
guardar(sg.head(12), "shap_global", "7.1 Importancia global agrupada (SHAP sobre el holdout)", dec=4)

fig, ax = plt.subplots(figsize=(8, 4.2))
NOMBRES = {"accommodates": "Capacidad de huéspedes", "calculated_host_listings_count": "Publicaciones del anfitrión",
           "bedrooms": "Dormitorios", "latitude": "Latitud", "banos": "Baños", "room_type": "Tipo de alojamiento",
           "minimum_nights": "Estadía mínima", "n_amenities": "Cantidad de amenities", "longitude": "Longitud",
           "neighbourhood_cleansed": "Barrio", "bano_compartido": "Baño compartido", "beds": "Camas"}
top = sg.head(10).iloc[::-1]
ax.barh([NOMBRES.get(i, i) for i in top.index], top["shap_medio_abs"], color=ROJO)
ax.set_xlabel("Contribución media absoluta al log(precio)")
ax.set_title("Qué pesa en el precio esperado (importancia SHAP)")
plt.tight_layout(); plt.savefig(RES_DIR / "fig_shap_global.png", dpi=150); plt.close()


7. Interpretabilidad: que explica el precio esperado

7.1 Importancia global agrupada (SHAP sobre el holdout)
                                shap_medio_abs     pct
accommodates                            0.1008 14.0802
calculated_host_listings_count          0.0966 13.4902
bedrooms                                0.0908 12.6756
latitude                                0.0862 12.0413
banos                                   0.0708  9.8925
room_type                               0.0649  9.0643
minimum_nights                          0.0591  8.2528
n_amenities                             0.0548  7.6530
longitude                               0.0372  5.1918
neighbourhood_cleansed                  0.0290  4.0527
bano_compartido                         0.0186  2.6012
beds                                    0.0067  0.9387


## 8. Profundizacion: brecha, demanda y etica

La brecha se define como log(precio real) menos log(precio esperado), de modo que cobrar
la mitad y cobrar el doble queden a igual distancia del cero.

Se analiza como se reparte el senalamiento entre barrios y grupos, se muestran casos
individuales en el lenguaje que pide el negocio, y se mide la asociacion entre la brecha
y la actividad reciente de resenas.

**Advertencia central:** las resenas son una aproximacion a la demanda, no un registro de
reservas, y el dataset es una foto de un momento. Lo que sigue documenta una **asociacion**;
no permite concluir que un precio por encima del esperado **cause** menor demanda. Hay al
menos tres explicaciones alternativas: calidad no observada, causalidad inversa (un
anfitrion sin reservas puede bajar el precio) y el desfase temporal entre el precio de
junio de 2026 y resenas de los doce meses previos.

In [9]:
titulo("8. Profundizacion: brecha, demanda y etica")
# 8.1 brecha y KPI sobre el holdout, con el umbral fijado ex ante
brecha = yh - pred_hold[GANADOR]           # log(precio real) - log(precio esperado)
pct = np.expm1(brecha) * 100
sub, sob = brecha < -UMBRAL_LOG, brecha > UMBRAL_LOG
err_med = np.median(np.abs(brecha))
out(f"error mediano del modelo en el holdout: {err_med:.4f} en log "
    f"(+{np.expm1(err_med)*100:.1f}% / {np.expm1(-err_med)*100:.1f}%)")
out(f"umbral fijado ex ante: {UMBRAL_LOG} en log "
    f"(+{np.expm1(UMBRAL_LOG)*100:.1f}% / {np.expm1(-UMBRAL_LOG)*100:.1f}%)")
kpi = pd.DataFrame({"publicaciones": [len(hold)], "pct_con_desvio": [(sub | sob).mean() * 100],
                    "pct_subvaluadas": [sub.mean() * 100], "pct_sobrevaluadas": [sob.mean() * 100]},
                   index=["holdout"])
guardar(kpi, "kpi_desvio", "8.1 KPI de desvio sobre el holdout", dec=1)

kb = pd.DataFrame({"barrio": hold["neighbourhood_cleansed"], "sub": sub, "sob": sob})
top_b = kb["barrio"].value_counts().head(10).index
tb = kb[kb["barrio"].isin(top_b)].groupby("barrio").agg(
    publicaciones=("sub", "size"), pct_subvaluadas=("sub", lambda s: s.mean() * 100),
    pct_sobrevaluadas=("sob", lambda s: s.mean() * 100))
tb["pct_con_desvio"] = tb["pct_subvaluadas"] + tb["pct_sobrevaluadas"]
guardar(tb.sort_values("publicaciones", ascending=False), "kpi_por_barrio",
        "8.2 KPI por barrio (diez barrios con mas publicaciones en el holdout)", dec=1)

# 8.3 casos individuales, en el lenguaje que pidio el tutor
casos = hold.assign(precio_esperado=np.exp(pred_hold[GANADOR]), brecha_pct=pct)
cols_caso = ["id", "neighbourhood_cleansed", "room_type", "accommodates",
             "bedrooms", "banos", "price", "precio_esperado", "brecha_pct"]
def tipicos(mascara, n=4):
    """Casos representativos: los mas cercanos a la brecha mediana del grupo senalado.
    Los extremos no sirven de ejemplo porque suelen ser errores de carga."""
    sub_df = casos.loc[mascara, cols_caso]
    med = sub_df["brecha_pct"].median()
    return sub_df.reindex((sub_df["brecha_pct"] - med).abs().sort_values().index).head(n)

guardar(pd.concat([tipicos(sub), tipicos(sob)]).set_index("id"), "casos_ejemplo",
        "8.3 Ejemplos tipicos: 'esta publicacion esta X% por encima o por debajo del precio esperado'\n"
        "(los cuatro mas representativos de cada grupo senalado)", dec=1)

# los extremos son sospechosos de error de carga, no de decision de precio
extremos = casos.loc[sob & (pct > 500), cols_caso]
out(f"\nPublicaciones senaladas con brecha mayor a +500%: {len(extremos)} "
    f"({len(extremos)/len(hold)*100:.2f}% del holdout). Precio mediano publicado: "
    f"{extremos['price'].median():,.0f} ARS contra {hold['price'].median():,.0f} del holdout.")
out("Son candidatas a error de carga (por ejemplo, precio mensual cargado como precio por")
out("noche) mas que a decisiones de precio, y se reportan como hallazgo de calidad de datos.")
guardar(extremos.nlargest(5, "brecha_pct").set_index("id"), "calidad_extremos",
        "Cinco publicaciones con la brecha mas extrema (sospechosas de error de carga)", dec=1)

# 8.4 asociacion entre brecha y demanda reciente (NO causalidad)
ltm = hold["number_of_reviews_ltm"].to_numpy()
fr = pd.to_datetime(hold["first_review"], errors="coerce")
variantes = {"todas": np.ones(len(hold), bool),
             "con al menos una resena": (hold["number_of_reviews"] > 0).to_numpy(),
             "primera resena hace mas de 12 meses": (fr <= SNAPSHOT - pd.Timedelta(days=365)).to_numpy()}
filas, por_decil = [], {}
for nombre_v, mask in variantes.items():
    b_, y_ = brecha[mask], ltm[mask]
    dec_ = pd.qcut(b_, 10, labels=False, duplicates="drop")
    d1, d10 = y_[dec_ == 0], y_[dec_ == dec_.max()]
    rho, p_rho = stats.spearmanr(b_, y_)
    p_mw = stats.mannwhitneyu(d1, d10, alternative="greater").pvalue
    filas.append({"variante": nombre_v, "publicaciones": int(mask.sum()), "spearman": rho,
                  "p_spearman": p_rho, "ltm_mediana_decil1": np.median(d1),
                  "ltm_mediana_decil10": np.median(d10), "p_mann_whitney": p_mw})
    por_decil[nombre_v] = pd.Series(y_).groupby(dec_).median()
guardar(pd.DataFrame(filas).set_index("variante"), "demanda_asociacion",
        "8.4 Asociacion entre brecha de precio y actividad reciente (holdout)", dec=4)
guardar(pd.DataFrame(por_decil), "demanda_deciles",
        "Mediana de resenas de 12 meses por decil de brecha (1 = mas por debajo)", dec=1)

fig, ax = plt.subplots(figsize=(9, 3.8))
sd = por_decil["todas"]
ax.bar(sd.index + 1, sd.values, color=ROJO)
ax.set_xticks(range(1, len(sd) + 1))
ax.set_xlabel("Decil de brecha de precio (1 = más por debajo del esperado, 10 = más por encima)")
ax.set_ylabel("Reseñas en 12 meses (mediana)")
ax.set_title("Actividad reciente según el posicionamiento del precio (asociación, no causalidad)")
plt.tight_layout(); plt.savefig(RES_DIR / "fig_demanda_deciles.png", dpi=150); plt.close()

# 8.5 control por calidad observable: la asociacion dentro de estratos de calificacion
if "review_scores_rating" in hold.columns:
    r = hold["review_scores_rating"].to_numpy()
    valido = ~np.isnan(r)
    filas = []
    if valido.sum() > 500:
        # la calificacion esta muy concentrada cerca de 5: se usan cortes por cuantil
        # y se etiqueta cada estrato con su rango real
        codigos = pd.qcut(r[valido], 3, labels=False, duplicates="drop")
        for nivel in sorted(pd.unique(codigos)):
            m = np.zeros(len(hold), bool)
            sel = np.where(valido)[0][codigos == nivel]
            m[sel] = True
            rho, p_rho = stats.spearmanr(brecha[m], ltm[m])
            filas.append({"estrato": f"calificacion {r[sel].min():.2f} a {r[sel].max():.2f}",
                          "publicaciones": int(m.sum()), "spearman": rho, "p_spearman": p_rho})
    guardar(pd.DataFrame(filas).set_index("estrato"), "demanda_por_calificacion",
            "8.5 La asociacion dentro de estratos de calificacion (control por calidad observable)", dec=4)

# 8.6 ventanas de resenas recientes
path_rev = buscar_csv("reviews.csv.gz", "reviews*.csv.gz", "reviews.csv")
if path_rev is not None:
    rev = pd.read_csv(path_rev, usecols=["listing_id", "date"], parse_dates=["date"])
    out(f"\nreviews: {path_rev} | {len(rev):,} resenas")
    filas = []
    for dias in (90, 180, 365):
        corte = SNAPSHOT - pd.Timedelta(days=dias)
        cnt = rev[rev["date"] >= corte].groupby("listing_id").size()
        v = hold["id"].map(cnt).fillna(0).to_numpy()
        rho, p_rho = stats.spearmanr(brecha, v)
        filas.append({"ventana_dias": dias, "resenas_totales": int(cnt.sum()),
                      "spearman": rho, "p_spearman": p_rho})
    rho, p_rho = stats.spearmanr(brecha, ltm)
    filas.append({"ventana_dias": "365 (number_of_reviews_ltm)", "resenas_totales": int(ltm.sum()),
                  "spearman": rho, "p_spearman": p_rho})
    guardar(pd.DataFrame(filas).set_index("ventana_dias"), "demanda_ventanas",
            "8.6 Asociacion segun la ventana temporal de resenas", dec=4)
else:
    out("\nreviews.csv.gz no disponible: la seccion de ventanas temporales queda pendiente.")

# 8.7 auditoria etica
aud = []
for col, etiqueta in [("host_is_superhost", "superanfitrion"), ("room_type", "tipo de habitacion")]:
    if col not in hold.columns:
        continue
    g = pd.DataFrame({"grupo": hold[col].astype(str), "sub": sub, "sob": sob, "brecha": brecha})
    t = g.groupby("grupo").agg(publicaciones=("sub", "size"),
                               pct_subvaluadas=("sub", lambda s: s.mean() * 100),
                               pct_sobrevaluadas=("sob", lambda s: s.mean() * 100),
                               brecha_mediana_pct=("brecha", lambda s: np.expm1(np.median(s)) * 100))
    t.insert(0, "atributo", etiqueta)
    aud.append(t[t["publicaciones"] >= 30])
if aud:
    guardar(pd.concat(aud), "auditoria_etica",
            "8.7 Auditoria: como se reparte el senalamiento entre grupos", dec=1)

# multi-propiedad: el modelo usa calculated_host_listings_count como variable
g = pd.DataFrame({"comercial": np.where(hold["calculated_host_listings_count"] > 2,
                                        "anfitrion con mas de 2", "anfitrion con 1 o 2"),
                  "sub": sub, "sob": sob})
guardar(g.groupby("comercial").agg(publicaciones=("sub", "size"),
                                   pct_subvaluadas=("sub", lambda s: s.mean() * 100),
                                   pct_sobrevaluadas=("sob", lambda s: s.mean() * 100)),
        "auditoria_escala_anfitrion", "Senalamiento segun escala del anfitrion", dec=1)


8. Profundizacion: brecha, demanda y etica
error mediano del modelo en el holdout: 0.2123 en log (+23.6% / -19.1%)
umbral fijado ex ante: 0.4253 en log (+53.0% / -34.6%)

8.1 KPI de desvio sobre el holdout
         publicaciones  pct_con_desvio  pct_subvaluadas  pct_sobrevaluadas
holdout           5721            20.5             11.3                9.2

8.2 KPI por barrio (diez barrios con mas publicaciones en el holdout)
              publicaciones  pct_subvaluadas  pct_sobrevaluadas  pct_con_desvio
barrio                                                                         
Palermo                2018             12.5                9.1            21.6
Recoleta                793             10.8               11.0            21.8
San Nicolas             477              4.6               10.7            15.3
Retiro                  282             11.7                6.4            18.1
Monserrat               242              7.0                5.0            12.0
Belgrano    

,publicaciones,pct_subvaluadas,pct_sobrevaluadas
comercial,,,
anfitrion con 1 o 2,2656,10.090361,8.471386
anfitrion con mas de 2,3065,12.267537,9.885808


## 9. Traduccion a negocio

Escenario **parametrico**: cuantifica la magnitud agregada de la brecha bajo supuestos
explicitos de ocupacion y de cuanta brecha decide cerrar el anfitrion.

No es una proyeccion de ingresos recuperables. Hacerlo supondria que bajar el precio
aumenta la demanda, que es exactamente lo que los datos no permiten afirmar.

In [10]:
titulo("9. Traduccion a negocio: escenario parametrico")
out("El escenario NO supone que corregir el precio aumente la demanda: esa relacion")
out("es asociativa y no fue demostrada. Cuantifica la magnitud de la brecha de")
out("posicionamiento bajo supuestos explicitos que el lector puede cambiar.")
noches = hold["number_of_reviews_ltm"].to_numpy()  # proxy de actividad, no de reservas
filas = []
for tasa in (0.3, 0.5, 0.7):          # fraccion de la brecha que el anfitrion decide cerrar
    for noches_mes in (5, 10, 15):     # noches ocupadas por mes, supuesto explicito
        dif = np.exp(pred_hold[GANADOR])[sub] - ph_real[sub]
        monto = (dif * tasa * noches_mes * 12).sum()
        filas.append({"fraccion_de_brecha_cerrada": tasa, "noches_ocupadas_por_mes": noches_mes,
                      "publicaciones_subvaluadas": int(sub.sum()),
                      "brecha_anual_ARS_millones": monto / 1e6})
guardar(pd.DataFrame(filas), "escenario_economico",
        "9.1 Magnitud de la brecha en publicaciones subvaluadas del holdout", dec=1, index=False)
out("\nLectura: son ordenes de magnitud sobre el 20% del mercado analizado, bajo supuestos")
out("de ocupacion que el dataset no observa. No es una proyeccion de ingresos.")


9. Traduccion a negocio: escenario parametrico
El escenario NO supone que corregir el precio aumente la demanda: esa relacion
es asociativa y no fue demostrada. Cuantifica la magnitud de la brecha de
posicionamiento bajo supuestos explicitos que el lector puede cambiar.

9.1 Magnitud de la brecha en publicaciones subvaluadas del holdout
   fraccion_de_brecha_cerrada  noches_ocupadas_por_mes  publicaciones_subvaluadas  brecha_anual_ARS_millones
0                         0.3                        5                        644                      704.7
1                         0.3                       10                        644                    1,409.4
2                         0.3                       15                        644                    2,114.1
3                         0.5                        5                        644                    1,174.5
4                         0.5                       10                        644                    2,349.0
5     

## 10. Resumen de resultados

In [11]:
titulo("10. Resumen")
resumen = pd.DataFrame([
    {"indicador": "Publicaciones del universo", "valor": f"{len(u):,}"},
    {"indicador": "Anfitriones", "valor": f"{u['host_id'].nunique():,}"},
    {"indicador": "Holdout (publicaciones / anfitriones)", "valor": f"{len(hold):,} / {hold['host_id'].nunique():,}"},
    {"indicador": "Modelo elegido", "valor": GANADOR},
    {"indicador": "MdAPE benchmark (holdout)", "valor": f"{hm.loc['Benchmark mediana','MdAPE']:.1f}%"},
    {"indicador": "MdAPE regresion lineal (holdout)", "valor": f"{hm.loc['Regresion lineal','MdAPE']:.1f}%"},
    {"indicador": f"MdAPE {GANADOR} (holdout)", "valor": f"{hm.loc[GANADOR,'MdAPE']:.1f}%"},
    {"indicador": "Reduccion sobre el benchmark", "valor": f"{hm.loc[GANADOR,'reduccion_vs_benchmark_pct']:.1f}%"},
    {"indicador": "R2 sobre log(precio) (holdout)", "valor": f"{hm.loc[GANADOR,'R2_log']:.3f}"},
    {"indicador": "Publicaciones con desvio significativo", "valor": f"{(sub|sob).mean()*100:.1f}%"},
    {"indicador": "Spearman brecha vs actividad reciente", "valor": f"{stats.spearmanr(brecha, ltm)[0]:+.3f}"},
])
guardar(resumen.set_index("indicador"), "resumen", "10.1 Numeros principales del proyecto", index=True)

out(f"\ntiempo total: {(time.time()-T0)/60:.1f} min")
(RES_DIR / "log_ejecucion.txt").write_text("\n".join(BUF), encoding="utf-8")
print(f"\nLog completo en {(RES_DIR / 'log_ejecucion.txt').resolve()}")


10. Resumen

10.1 Numeros principales del proyecto
                                                valor
indicador                                            
Publicaciones del universo                     27,287
Anfitriones                                    13,432
Holdout (publicaciones / anfitriones)   5,721 / 2,687
Modelo elegido                                XGBoost
MdAPE benchmark (holdout)                       28.3%
MdAPE regresion lineal (holdout)                24.5%
MdAPE XGBoost (holdout)                         21.5%
Reduccion sobre el benchmark                    24.1%
R2 sobre log(precio) (holdout)                  0.506
Publicaciones con desvio significativo          20.5%
Spearman brecha vs actividad reciente          -0.145

tiempo total: 2.0 min

Log completo en C:\Users\sibon\Documents\github\TP_final_999201523_Siboni_Romero\results\log_ejecucion.txt
